# VerseLens — Poem Interpretation Fine-Tuning (QLoRA)

**Goal:** fine-tune `Qwen/Qwen2.5-1.5B-Instruct` to generate detailed, text-grounded interpretations of complete poems, using `poemsum_train(1).csv`.

**Training:** Google Colab GPU + 4-bit QLoRA.  
**Deployment goal:** export the LoRA adapter so it can be used on your MacBook Air M3 (16 GB). Your existing theme/emotion classifier is not changed by this notebook.

### Important quality safeguards
- Keep poems together as whole examples; this is **not** line-by-line training.
- Split by normalized poem text into train/validation/test groups.
- Use validation loss to select the best checkpoint; reserve test examples for final evaluation.
- Inspect generated interpretations manually. A lower loss does not guarantee a correct literary interpretation.
- Long poems and reference interpretations are capped at token limits, so some long examples are truncated.

## 1. Set up Colab

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or a better available GPU). Then run the cells in order.

The installation cell pins compatible major-version ranges rather than freezing every dependency to a brittle exact version. The smaller model and 2-epoch starter setup are intended to be faster, but exact training time cannot be guaranteed.

In [ ]:
!pip -q install -U "transformers>=4.51.0,<5" "peft>=0.15.0" "accelerate>=1.2.0" "bitsandbytes>=0.45.0" "datasets>=3.2.0" "scikit-learn>=1.5.0" sentencepiece

In [ ]:
import os, re, json, math, random, shutil, zipfile, unicodedata, hashlib, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("Enable a GPU runtime in Colab: Runtime → Change runtime type → GPU.")

## 2. Upload the dataset

Upload `poemsum_train(1).csv` when prompted. The notebook expects these columns:
- `ctext`: complete poem text (model input)
- `text`: reference interpretation (training target)
- `Title`, `Poet`: optional metadata included in the prompt
- `Poem Link`: provenance only; never fed to the model

In [ ]:
DATA_PATH = "/content/poemsum_train(1).csv"

if not os.path.exists(DATA_PATH):
    from google.colab import files
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith(".csv")]
    if not candidates:
        raise FileNotFoundError("Upload poemsum_train(1).csv")
    # Prefer the expected filename; otherwise use the first uploaded CSV.
    chosen = next((n for n in candidates if "poemsum_train" in n.lower()), candidates[0])
    DATA_PATH = f"/content/{chosen}"

df = pd.read_csv(DATA_PATH)
required = {"ctext", "text"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found: {df.columns.tolist()}")

for col in ["Title", "Poet", "ctext", "text"]:
    if col not in df.columns:
        df[col] = ""

for col in ["Title", "Poet", "ctext", "text"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

df = df[(df["ctext"].str.len() > 20) & (df["text"].str.len() > 80)].copy()
df = df.reset_index(drop=True)

print("Rows kept:", len(df))
print("Columns:", df.columns.tolist())
print("Missing values in required fields:", df[["ctext", "text"]].eq("").sum().to_dict())
print("\nCharacter lengths:")
display(df[["ctext", "text"]].apply(lambda s: s.str.len().describe()).round(1))
display(df[["Title", "Poet", "ctext", "text"]].head(3))

## 3. Inspect data and create leakage-resistant splits

The split groups exact normalized duplicate poem texts together so identical poems cannot appear in both training and evaluation. This does not detect every possible near-duplicate or different edition of the same poem, so inspect suspicious matches if evaluation quality is critical.

In [ ]:
def normalize_poem(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    return re.sub(r"\\s+", " ", s).strip()

df["group_key"] = df["ctext"].map(normalize_poem)
print("Rows:", len(df), "| unique normalized poem texts:", df["group_key"].nunique())

# 10% test holdout; then 10% of remaining for validation (~9% overall).
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
trainval_idx, test_idx = next(gss_test.split(df, groups=df["group_key"]))

trainval = df.iloc[trainval_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1111, random_state=SEED + 1)
train_idx, val_idx = next(gss_val.split(trainval, groups=trainval["group_key"]))

train_df = trainval.iloc[train_idx].reset_index(drop=True)
val_df = trainval.iloc[val_idx].reset_index(drop=True)

train_groups = set(train_df["group_key"])
val_groups = set(val_df["group_key"])
test_groups = set(test_df["group_key"])
assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
print("Group overlap checks: passed")

## 4. Configure the model and QLoRA

We use `Qwen/Qwen2.5-1.5B-Instruct`, a smaller instruction model chosen to reduce training time and make local deployment easier. QLoRA keeps base weights in 4-bit and learns LoRA adapters. Runtime still depends on the assigned Colab GPU and actual token lengths.

To reduce training time, token budgets are set to up to 1,024 prompt tokens and 384 target tokens (1,408 total). Long prompts keep both their beginning and ending. This may truncate some longer poems or reference interpretations. The preprocessing keeps the beginning and end of unusually long prompts, which helps retain both the opening and ending of long poems.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "/content/verselens_qwen2_5_1_5b_interpreter"
ADAPTER_DIR = f"{OUTPUT_DIR}/adapter"
MAX_PROMPT_TOKENS = 1024
MAX_TARGET_TOKENS = 384
MAX_SEQ_LENGTH = MAX_PROMPT_TOKENS + MAX_TARGET_TOKENS

# Faster starter configuration for a Colab T4. Start with 2 epochs; compare validation/test outputs before considering more training.
NUM_EPOCHS = 2
LEARNING_RATE = 5e-5
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

os.makedirs(OUTPUT_DIR, exist_ok=True)

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    Trainer, TrainingArguments, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

compute_dtype = torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=compute_dtype,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Format whole-poem → interpretation examples

The reference `text` column is the target. The prompt asks for a careful literary interpretation grounded in the poem, but we do **not** fabricate explicit emotion/theme labels from this dataset because it has no structured labels for them.

In [ ]:
SYSTEM_PROMPT = (
    "You are VerseLens, a careful literary analyst. Interpret the complete poem using "
    "only evidence supported by its wording. Explain the central meaning, themes, imagery, "
    "symbols, tone, emotional tensions, and how the poem develops when the text supports "
    "those points. Distinguish interpretation from certainty; do not invent biographical "
    "facts, historical context, or authorial intent. Write a detailed, coherent explanation."
)

def make_prompt(row):
    title = row["Title"].strip()
    poet = row["Poet"].strip()
    poem = row["ctext"].strip()
    metadata = []
    if title:
        metadata.append(f"Title: {title}")
    if poet:
        metadata.append(f"Poet: {poet}")
    metadata_text = "\n".join(metadata)
    if metadata_text:
        user_text = f"{metadata_text}\n\nPoem:\n{poem}\n\nWrite a detailed interpretation of this poem."
    else:
        user_text = f"Poem:\n{poem}\n\nWrite a detailed interpretation of this poem."
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def truncate_keep_ends(ids, max_len):
    if len(ids) <= max_len:
        return ids
    head = int(max_len * 0.75)
    tail = max_len - head
    return ids[:head] + ids[-tail:]

def encode_row(row):
    prompt_text = make_prompt(row)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    prompt_ids = truncate_keep_ends(prompt_ids, MAX_PROMPT_TOKENS)

    target_text = row["text"].strip()
    target_ids = tokenizer(target_text, add_special_tokens=False)["input_ids"]
    target_ids = target_ids[:MAX_TARGET_TOKENS - 1] + [tokenizer.eos_token_id]

    input_ids = prompt_ids + target_ids
    labels = ([-100] * len(prompt_ids)) + target_ids.copy()
    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

from torch.utils.data import Dataset

class PoemInterpretationDataset(Dataset):
    def __init__(self, frame):
        self.examples = [encode_row(row) for row in frame.to_dict("records")]
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        return self.examples[idx]

train_dataset = PoemInterpretationDataset(train_df)
val_dataset = PoemInterpretationDataset(val_df)
test_dataset = PoemInterpretationDataset(test_df)

print("Tokenized examples:", len(train_dataset), len(val_dataset), len(test_dataset))
print("First example token count:", len(train_dataset[0]["input_ids"]))

## 6. Train with validation and keep the best checkpoint

The adapter is trained only on target interpretation tokens; prompt tokens are masked out of the loss. The trainer evaluates after each epoch and restores the checkpoint with the best validation loss.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    group_by_length=True,
    max_grad_norm=0.3,
    seed=SEED,
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    tokenizer=tokenizer,
)

train_result = trainer.train()
print(train_result)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

## 7. Evaluate on the held-out test set

Test loss is a useful signal, not a complete quality score. We also generate interpretations for a few held-out poems so you can manually inspect accuracy, evidence use, coherence, and hallucinations.

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
print(json.dumps(test_metrics, indent=2, default=float))
with open(f"{OUTPUT_DIR}/test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2, default=float)

In [ ]:
def generate_interpretation(row, max_new_tokens=512, temperature=0.3):
    prompt_text = make_prompt(row)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    prompt_ids = truncate_keep_ends(prompt_ids, MAX_PROMPT_TOKENS)
    inputs = torch.tensor([prompt_ids], dtype=torch.long, device=model.device)
    attention_mask = torch.ones_like(inputs)

    model.eval()
    model.config.use_cache = True
    with torch.no_grad():
        output = model.generate(
            input_ids=inputs,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=max(temperature, 1e-5),
            top_p=0.9,
            repetition_penalty=1.08,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output[0, inputs.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

sample_n = min(3, len(test_df))
for idx in range(sample_n):
    row = test_df.iloc[idx]
    print("=" * 100)
    print("TITLE:", row["Title"], "| POET:", row["Poet"])
    print("\nPOEM (first 1200 chars):\n", row["ctext"][:1200])
    print("\nREFERENCE INTERPRETATION (first 1600 chars):\n", row["text"][:1600])
    print("\nMODEL INTERPRETATION:\n", generate_interpretation(row))

## 8. Save adapter, tokenizer, config, and evaluation results

This exports the LoRA adapter rather than the full base model. The local Mac must load the exact same `Qwen/Qwen2.5-1.5B-Instruct` base model plus this adapter. Do not rename the adapter files inside the folder.

In [ ]:
ADAPTER_DIR = f"{OUTPUT_DIR}/adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

metadata = {
    "base_model": MODEL_ID,
    "method": "QLoRA (4-bit NF4) + PEFT LoRA",
    "dataset": os.path.basename(DATA_PATH),
    "input_column": "ctext",
    "target_column": "text",
    "rows_after_filtering": int(len(df)),
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "max_prompt_tokens": MAX_PROMPT_TOKENS,
    "max_target_tokens": MAX_TARGET_TOKENS,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "seed": SEED,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "test_metrics": test_metrics,
}
with open(f"{ADAPTER_DIR}/verselens_training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=float)

ZIP_PATH = shutil.make_archive("/content/verselens_qwen2_5_1_5b_interpreter_adapter", "zip", ADAPTER_DIR)
print("Adapter ZIP:", ZIP_PATH, "| size MiB:", round(os.path.getsize(ZIP_PATH) / 1024**2, 2))

## 9. Copy the adapter ZIP to Google Drive

Run this cell to save the exported adapter beyond the current Colab session. A Drive authorization prompt will appear.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DEST = "/content/drive/MyDrive/verselens_qwen2_5_1_5b_interpreter_adapter.zip"
shutil.copy2(ZIP_PATH, DRIVE_DEST)
print("Saved to:", DRIVE_DEST)

## 10. Local Mac inference (run separately on your Mac)

After downloading and extracting `verselens_qwen2_5_1_5b_interpreter_adapter.zip` on your Mac, create a Python virtual environment and install the packages below. The 1.5B base model is much lighter than the previous 4B option. For local use on a 16 GB Mac, start with a modest context/output length; for the smallest runtime footprint, we can later convert to a supported quantized MLX/GGUF format.

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install "transformers>=4.51,<5" "peft>=0.15" accelerate torch sentencepiece
```

Place the extracted adapter folder at `./verselens_qwen2_5_1_5b_interpreter_adapter/`, then run:

In [ ]:
# LOCAL MAC INFERENCE — run in your Mac project, not in the Colab runtime.
# If running inside Colab, skip this cell.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "./verselens_qwen2_5_1_5b_interpreter_adapter"

SYSTEM_PROMPT = (
    "You are VerseLens, a careful literary analyst. Interpret the complete poem using "
    "only evidence supported by its wording. Explain the central meaning, themes, imagery, "
    "symbols, tone, emotional tensions, and how the poem develops when the text supports "
    "those points. Distinguish interpretation from certainty; do not invent biographical "
    "facts, historical context, or authorial intent. Write a detailed, coherent explanation."
)

local_device = "mps" if torch.backends.mps.is_available() else "cpu"
local_dtype = torch.float16 if local_device == "mps" else torch.float32

local_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=local_dtype,
    low_cpu_mem_usage=True,
)
local_model = PeftModel.from_pretrained(base, ADAPTER_PATH)
local_model.to(local_device)
local_model.eval()

def interpret_poem_local(poem, title="", poet="", max_new_tokens=400):
    meta = []
    if title.strip():
        meta.append(f"Title: {title.strip()}")
    if poet.strip():
        meta.append(f"Poet: {poet.strip()}")
    user_text = ("\n".join(meta) + "\n\n" if meta else "") + (
        "Poem:\n" + poem.strip() + "\n\nWrite a detailed interpretation of this poem."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    prompt = local_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = local_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536)
    inputs = {k: v.to(local_device) for k, v in inputs.items()}
    with torch.no_grad():
        out = local_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.3, top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=local_tokenizer.pad_token_id or local_tokenizer.eos_token_id,
            eos_token_id=local_tokenizer.eos_token_id,
        )
    return local_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Example:
# print(interpret_poem_local("Paste your complete poem here", title="Optional title", poet="Optional poet"))

## Before calling it done

- Check the training and test loss.
- Read the generated examples beside the reference interpretations.
- Look for unsupported claims, generic analysis, repetitive language, and failure to discuss actual wording.
- Keep the original classifier untouched; integrate this model as a separate interpretation component.
- This dataset's descriptions are reference interpretations, not objective ground truth. Human review is necessary.